# 课后练习解答（04.07_latency_comparison_and_profiling）

本解答对应《延迟对比与 MindStudio Profiling》课后练习，共 15 题。


### 问题1（单选题）

**题目：** CPU NMS algorithm latency 与 ACLNN runner wall time 的关系最准确的是？

A. 二者完全等价
B. runner wall time 包含编译、数据生成、ACL 初始化、文件 I/O 等端到端开销，不等于 kernel 纯耗时
C. CPU latency 一定比 NPU kernel 慢
D. 二者都不能测量

**解答：** B

**解析：** 实验中反复强调不能把 runner 端到端耗时当成 NPU kernel 纯耗时。


### 问题2（单选题）

**题目：** 想查看自定义算子 `YoloNmsCustom` 的真实任务耗时，应优先查看哪类 Profiling 输出？

A. op_summary_*.csv 或 MindStudio OP Info
B. README 标题
C. Git status
D. requirements.txt

**解答：** A

**解析：** op_summary 中可以找到算子名和 Task Duration(us)。


### 问题3（单选题）

**题目：** MSPROF 采集数据后，常见输出目录名前缀是？

A. PROF_
B. VOC_
C. JPEG_
D. GIT_

**解答：** A

**解析：** Ascend Profiling 目录通常以 PROF_ 加时间戳等信息命名。


### 问题4（单选题）

**题目：** 如果 Profiling 中 YoloNmsCustom 的 Task Duration 为 50.726 us，换算为毫秒约为？

A. 0.0507 ms
B. 50.726 ms
C. 50726 ms
D. 5.0726 s

**解答：** A

**解析：** 1000 微秒等于 1 毫秒。


### 问题5（多选题）

**题目：** 撰写延迟对比结论时，应明确区分哪些时间？

A. CPU NMS 算法耗时
B. ACLNN runner 端到端耗时
C. Profiling 中 kernel/task 耗时
D. 数据准备和编译开销

**解答：** A、B、C、D

**解析：** 不同层级的时间含义不同，混在一起会误导结论。


### 问题6（多选题）

**题目：** MindStudio Profiling / msprof 可以帮助观察哪些方面？

A. API 调用
B. Task Time
C. 算子统计
D. Host 侧瓶颈

**解答：** A、B、C、D

**解析：** 这些信息共同帮助定位端到端性能问题。


### 问题7（多选题）

**题目：** 如果 runner wall time 很长，可能包含哪些非 kernel 开销？

A. CMake/Make 编译
B. 输入文件生成和读取
C. ACL 初始化和同步
D. 输出文件写入

**解答：** A、B、C、D

**解析：** 这些都会让端到端 wall time 大于单个算子耗时。


### 问题8（判断题）

**题目：** 看到 runner wall time 比 CPU NMS 大，就能直接说明 NPU 自定义算子性能更差。

**解答：** 错误

**解析：** runner wall time 不是纯 kernel 时间，必须结合 Profiling 的 Task Duration 判断。


### 问题9（判断题）

**题目：** Profiling 结果中若能找到 YoloNmsCustom 行，说明自定义算子进入了被采集的执行链路。

**解答：** 正确

**解析：** 这是验证自定义算子被实际调用的重要证据。


### 问题10（填空题）

**题目：** 算子耗时字段 `Task Duration(us)` 的单位是 `____`。

**解答：** 微秒

**解析：** us 表示 microseconds。


### 问题11（填空题）

**题目：** 本实验用于采集 Profiling 的命令行工具是 `____`。

**解答：** msprof

**解析：** MindStudio GUI 可打开 msprof 生成的 PROF 数据进一步分析。


### 问题12（简答题）

**题目：** 为什么实验报告中不建议写“runner 14.59s，所以 NPU 比 CPU 慢”？

**解答：** 因为 runner 时间包含脚本启动、CMake/Make、ACL 初始化、输入输出文件读写和同步等大量固定开销，不代表 NPU kernel 纯执行时间。应读取 Profiling 中 YoloNmsCustom 的 Task Duration 再下结论。

**解析：** 这是本章最重要的性能分析边界。


### 问题13（简答题）

**题目：** MindStudio GUI 中分析自定义算子耗时的基本步骤是什么？

**解答：** 先用 msprof 生成 PROF 目录，再在 MindStudio Profiler 中导入该目录，进入 Timeline/OP Summary，搜索 YoloNmsCustom，查看 Task Duration、stream/task id 和调用位置。

**解析：** GUI 适合可视化观察时间线和算子统计。


### 问题14（简答题）

**题目：** 如果 Profiling 报告中找不到 YoloNmsCustom，可能是什么原因？

**解答：** 可能没有运行 ACLNN 自定义算子路径、OPP 未正确安装、采集命令包住了错误程序、程序走了 CPU fallback，或 Profiling 配置没有采集 task/op 信息。

**解析：** 需要先确认正确性验证中 custom op 确实被调用。


### 问题15（代码设计题）

**题目：** 写一条 msprof 命令，对 PyACL YOLO 推理脚本采集 Profiling 数据。

**解答：**

```bash
msprof \
  --application="python src/scripts/pyacl_yolo_infer.py --config src/configs/yolo_edge.yaml --image src/data/images/bus.jpg --repeat 50" \
  --output=profiles/yolo_edge \
  --runtime-api=on \
  --task-time=on \
  --aicpu=on
```

**解析：** 实际参数可按 CANN/MSPROF 版本调整，核心是用 --application 包住被测命令。
